# Corrective loops across the benchmark suite

`sf.CorrectiveLoop` (the protocol from [this paper](https://openreview.net/pdf?id=XSIYfTm2h7))
runs a
panel of members against each Case, checks every draft mid-run on the Benchmark's advertised
check surface, and — when a draft fails — feeds the sanitized verification feedback through a
judge-coached rewrite, up to `max_rounds`. The best passing draft is submitted verbatim.

Every installed Benchmark advertises whether its check surface is free or paid:

| Benchmark | Checked by | Mid-run check cost |
|---|---|---|
| `ifeval` | vendored official verifier (deterministic) | free |
| `healthbench-worst30` | pinned GPT-5.4 rubric Judge | **paid — every round spends judge tokens** |
| `healthbench-professional` | the same pinned Judge | **paid — and 525 Cases, not 157** |
| `draco` | pinned Gemini rubric Judge | **paid — every round spends judge tokens** |

## Before running

From a terminal:

```bash
screamingface prepare --all  # first run only: download all three Benchmark assets
screamingface up             # start Gateway :9105, Scoreboard :9106, and Engine :9108
screamingface status
```

Use `screamingface logs` to inspect startup failures and `screamingface down` when finished.

For DRACO, export `TAVILY_API_KEY` before `screamingface up`: the answer routes use its guarded
tool loop, and the Engine fails before model spend when that retrieval mechanism is missing.


In [ ]:
import screamingface as sf

sf.connect()

## Define the panel and the judge

In [ ]:
ANSWER_PROMPT = (
    "Answer the request accurately and completely. "
    "Follow every instruction and formatting constraint in the request."
)

PARAMS = {"max_tokens": 16384, "temperature": 0.0}

deepseek = sf.Model(
    model="openrouter/deepseek/deepseek-v4-pro",
    prompt=ANSWER_PROMPT,
    params=PARAMS,
)
qwen = sf.Model(
    model="openrouter/qwen/qwen3.8-2.4t-a95b",
    prompt=ANSWER_PROMPT,
    params=PARAMS,
)
glm = sf.Model(
    model="openrouter/z-ai/glm-5.2",
    prompt=ANSWER_PROMPT,
    params=PARAMS,
)

In [ ]:
SYNTHESIS_PROMPT = (
    "Produce one final answer to the original request from the panel drafts. "
    "Preserve every instruction and formatting constraint."
)

kimi = sf.Model(
    model="openrouter/moonshotai/kimi-k3",
    prompt=SYNTHESIS_PROMPT,
    params=PARAMS,
)

corrective_loop = sf.CorrectiveLoop(members=[deepseek, qwen, glm], judge=kimi, max_rounds=3)
corrective_loop

## 1. IFEval — free deterministic checks

A first-round pass costs the member drafts and nothing else; only correction rounds add spend.


In [ ]:
ifeval_report = sf.evaluate(corrective_loop, benchmark="ifeval", limit=1)
ifeval_report

## 2. HealthBench worst-30% — paid rubric checks

The physician-authored rubric is graded by the pinned Judge, so every round — including a
first-round pass — makes one judge call per draft.

In [ ]:
healthbench_report = sf.evaluate(corrective_loop, benchmark="healthbench-worst30", limit=1)
healthbench_report

## 3. DRACO — paid rubric checks

Research-quality prompts with weighted rubrics; the longest and most expensive of the three.


In [ ]:
draco_report = sf.evaluate(corrective_loop, benchmark="draco", limit=1)
draco_report

## 4. Send the scores to the Scoreboard

Publication takes the evaluated `CandidateResult` and submits the Benchmark's **native
score** exactly as the Engine graded it — fractional or negative values included — and the
Scoreboard stores and ranks it without recalculating. Opt-in so **Run All** never changes
the public Leaderboard.

In [ ]:
PUBLISH_RESULT = False

submissions = (
    [
        sf.leaderboards.submit(report.candidates.only)
        for report in (ifeval_report, healthbench_report, draco_report)
    ]
    if PUBLISH_RESULT
    else None
)
submissions